## Stage 3 - Dino Contrastive

### Data folder structure 

In [1]:
import os

for root, dirs, files in os.walk("JIGSAWS"):
    level = root.replace("JIGSAWS", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"    {indent}{f}")

JIGSAWS/
    .DS_Store
  Experimental_setup/
      .DS_Store
    Knot_Tying/
        .DS_Store
      Balanced/
        GestureClassification/
          OneTrialOut/
            25_Out/
              itr_44/
              itr_43/
              itr_17/
              itr_28/
              itr_10/
              itr_9/
              itr_26/
              itr_19/
              itr_21/
              itr_7/
              itr_42/
              itr_45/
              itr_6/
              itr_20/
              itr_27/
              itr_1/
              itr_18/
              itr_11/
              itr_8/
              itr_16/
              itr_29/
              itr_34/
              itr_33/
              itr_32/
              itr_35/
              itr_50/
              itr_13/
              itr_14/
              itr_22/
              itr_4/
              itr_3/
              itr_25/
              itr_49/
              itr_40/
              itr_47/
              itr_24/
              itr_2/
         

In [2]:
print(os.listdir("Cholec80"))
print(os.listdir("JIGSAWS"))

['.DS_Store', 'cholec80']
['.DS_Store', 'Experimental_setup', 'Knot_Tying', 'Suturing', 'Needle_Passing']


In [3]:
with open("JIGSAWS/Suturing/transcriptions/Suturing_B001.txt") as f:
    print(f.read())

80 219 G1 
220 370 G5 
371 590 G8 
591 660 G2 
661 954 G3 
955 1097 G8 
1098 1124 G2 
1125 1401 G3 
1402 1439 G2 
1440 1698 G8 
1699 1783 G2 
1784 2285 G3 
2286 2495 G6 
2496 2679 G4 
2680 2750 G2 
2751 2976 G3 
2977 3155 G6 
3156 3308 G4 
3309 3659 G2 
3660 4011 G3 
4012 4321 G8 
4322 4374 G2 
4375 4704 G3 
4705 4846 G6 
4847 4983 G4 
4984 5059 G2 
5060 5354 G3 
5355 5447 G6 
5448 5635 G11 



In [4]:
videos = os.listdir("JIGSAWS/Suturing/video/")
print(sorted(videos)[:5])
print(f"Total videos: {len(videos)}")

['Suturing_B001_capture1.avi', 'Suturing_B001_capture2.avi', 'Suturing_B002_capture1.avi', 'Suturing_B002_capture2.avi', 'Suturing_B003_capture1.avi']
Total videos: 78


### Dataloader

In [5]:
import torch 
from torch.utils.data import Dataset, DataLoader
import av
import numpy as np 
import os
from torchvision import transforms
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ensures deterministic behavior in multi-head attention
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# mapping gestures to numbers
GESTURE_MAP = {
    'G1':0,'G2':1,'G3':2,'G4':3,'G5':4,
    'G6':5,'G8':6,'G9':7,'G10':8,'G11':9
}

In [6]:
class JIGSAWSClipDataset(Dataset):
    def __init__(self, task='Suturing', split_trials=None, clip_len=8):
        # split_trials is the list of trial names to include like 'B001', 'B002'

        self.clip_len = clip_len
        self.clips = [] # will (store video_path, start, end, label)

        trans_dir = f"JIGSAWS/{task}/transcriptions/"
        video_dir = f"JIGSAWS/{task}/video/"

        # looping through every transcription file - one per video
        for fname in sorted(os.listdir(trans_dir)):
            if not fname.endswith('.txt'):
                continue

            # extract trial name
            trial = fname.replace(f'{task}_', '').replace('.txt', '')

            # test trials skipped
            if split_trials and trial not in split_trials:
                continue
            
            # we use capture 1 camera angle
            video_path = os.path.join(video_dir, f"{task}_{trial}_capture1.avi") 
            if not os.path.exists(video_path):
                continue

            # read transcription file
            with open(os.path.join(trans_dir, fname)) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 3: continue
                    start, end, gesture = int(parts[0]), int(parts[1]), parts[2]

                    # keeping only gestures we know and long enough to sample 8 frames
                    if gesture in GESTURE_MAP and (end-start)>clip_len:
                        self.clips.append((video_path, start, end, GESTURE_MAP[gesture]))

                
        # transform to match ResNet
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias = True),
            transforms.Normalize([0.485,0.456,0.406],
                                [0.229,0.224,0.225]) 
        ])
        print(f"    {task} {split_trials}: {len(self.clips)} gesture clips")

    
    def _load_frames(self, video_path, start, end):
        # pick clip_len frame indices evenly spread across the gesture segment

        frame_ids = set(np.linspace(start, end, self.clip_len, dtype=int).tolist())
        frames = []
        container = av.open(video_path) # open the video file
        for i, frame in enumerate(container.decode(video=0)):
            if i in frame_ids:
                img = frame.to_ndarray(format='rgb24') # converting to RGB numpy array
                frames.append(self.transform(img)) # resize + transform
            if len(frames) == self.clip_len:
                break # stop when we have all frames

        container.close()

        while len(frames) < self.clip_len:
            frames.append(frames[-1]) # pad the last frame if less

        return torch.stack(frames) # stack into (T, C, H, W) tensor

    def __len__(self): return len(self.clips)

    def __getitem__(self,idx):
        video_path, start, end, label = self.clips[idx]
        frames = self._load_frames(video_path, start, end)

        # for stage 2 we return all frames
        return frames, label            

In [7]:
# loading a sample to check

ds = JIGSAWSClipDataset(task="Suturing")
img, label = ds[0]
print(f"Sample shape: {img.shape}, label: {label}, gesture: {list(GESTURE_MAP.keys())[label]}")

    Suturing None: 793 gesture clips
Sample shape: torch.Size([8, 3, 224, 224]), label: 0, gesture: G1


### Train/test split

In [8]:
# get all trial names from the transciptions folder
all_trials = [f.replace('Suturing_', '').replace('.txt','')
            for f in os.listdir('JIGSAWS/Suturing/transcriptions/')
            if f.endswith('.txt')]
all_trials = sorted(all_trials)
print("All trials:", all_trials)

# first four trials for test rest train
# JIGSAWS uses leave-one-user-out i am doing it in later stages

test_trials = all_trials[:4]
train_trials = all_trials[4:]

print(f"Train: {train_trials}")
print(f"Test: {test_trials}")

#  create dataset objects
train_clip_ds = JIGSAWSClipDataset('Suturing', split_trials=train_trials)
test_clip_ds  = JIGSAWSClipDataset('Suturing', split_trials=test_trials)

# dataloadaer handles batching + shuffling
train_clip_loader = DataLoader(train_clip_ds, batch_size=16, shuffle=True, num_workers=0)
test_clip_loader = DataLoader(test_clip_ds, batch_size=16, shuffle=False, num_workers=0)
print(f"\nTrain batches: {len(train_clip_loader)}, Test batches: {len(test_clip_loader)}")

All trials: ['B001', 'B002', 'B003', 'B004', 'B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Train: ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Test: ['B001', 'B002', 'B003', 'B004']
    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Suturing ['B001', 'B002', 'B003

### RAFT for optical flow

In [9]:
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


# load pretrained RAFT-small
weights = Raft_Small_Weights.DEFAULT
raft = raft_small(weights=weights).to(device).eval()

# freeze raft as we use it only for feature extraction
for param in raft.parameters():
    param.requires_grad = False

print("Raft loaded successfully")
print(f"Device: {device}")

Raft loaded successfully
Device: mps


### Precomputing and caching optical flow for all runs

In [10]:
import os
import numpy as np
import av
import torch
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
from torchvision import transforms
from tqdm import tqdm
import time

FLOW_CACHE_DIR = "flow_cache"
os.makedirs(FLOW_CACHE_DIR, exist_ok=True)

# resize to match our RGB pipeline
resize = transforms.Compose([
    transforms.ToTensor(),      # (3, H, W) as expected but in [0, 1] float
    transforms.Resize((224,224), antialias=True)
])

def get_clip_frames(video_path, start, end, clip_len=8):
    # loading the clip frames we need from the gesture seg
    frame_ids = set(np.linspace(start, end, clip_len, dtype=int).tolist())
    frames = []
    container = av.open(video_path)
    for i, frame in enumerate(container.decode(video=0)):
        if i in frame_ids:
            frames.append(frame.to_ndarray(format='rgb24'))
        if len(frames) == clip_len:
            break
    container.close()
    # padding if video ended early
    while len(frames) < clip_len:
        frames.append(frames[-1])
    return frames # list of clip np arr (H, W, 3)

def compute_clip_flow(frames, raft, device):
    # compute flow between consec frames to produce clip_len-1 flow map
    flows = []
    for i in range(len(frames)-1):
        f1 = resize(frames[i]).unsqueeze(0).to(device) * 255.0
        f2 = resize(frames[i+1]).unsqueeze(0).to(device) * 255.0    # both (1, 3, 224, 224)
        with torch.no_grad():
            flow = raft(f1, f2)[-1].squeeze(0).cpu().numpy()    # (2, 224, 224)

        flows.append(flow)
    return np.stack(flows)      # (clip_len-1, 2, 224, 224)


# do for all tasks - generate clip keys and compute flow
tasks = ['Suturing', 'Knot_Tying', 'Needle_Passing']
all_clips = []

for task in tasks:
    trans_dir = f"JIGSAWS/{task}/transcriptions/"
    video_dir = f"JIGSAWS/{task}/video/"
    for fname in sorted(os.listdir(trans_dir)):
        if not fname.endswith('.txt'): 
            continue
        trial = fname.replace(f'{task}_','').replace('.txt', '')
        video_path = os.path.join(video_dir, f"{task}_{trial}_capture1.avi")
        if not os.path.exists(video_path): 
            continue
        with open(os.path.join(trans_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 3: 
                    continue
                start, end, gesture = int(parts[0]), int(parts[1]), parts[2]
                if gesture in GESTURE_MAP and (end - start) >= 8:
                    # generate a unique key per clip: task_trial_start_end.npy
                    key = f"{task}_{trial}_{start}_{end}.npy"
                    all_clips.append((video_path, start, end, key))

print(f"Total clips to cache: {len(all_clips)}")

# compute and cache flow
skipped = 0
computed = 0
total_start = time.time()


for video_path, start, end, key in tqdm(all_clips, desc="Caching flow"):
    cache_path = os.path.join(FLOW_CACHE_DIR, key)
    if os.path.exists(cache_path):
        skipped += 1
        continue
    frames = get_clip_frames(video_path, start, end, clip_len=8)
    flow = compute_clip_flow(frames, raft, device)
    np.save(cache_path, flow)
    computed += 1

total_time = time.time() - total_start
print(f"\nComputed: {computed}, Skipped (cached): {skipped}")
print(f"Total time: {total_time/60:.1f} min")

# verify cache
cache_files = sorted(os.listdir(FLOW_CACHE_DIR))
sample      = np.load(os.path.join(FLOW_CACHE_DIR, cache_files[0]))
total_mb    = sum(os.path.getsize(os.path.join(FLOW_CACHE_DIR, f)) for f in cache_files) / 1e6

print(f"\nCached flow files: {len(cache_files)}")
print(f"Sample: {cache_files[0]}")
print(f"Shape:  {sample.shape}  (clip_len-1, 2, H, W)")
print(f"Min: {sample.min():.3f}, Max: {sample.max():.3f}")
print(f"Total cache size: {total_mb:.1f} MB")

Total clips to cache: 1387


Caching flow: 100%|██████████| 1387/1387 [00:00<00:00, 394273.10it/s]


Computed: 0, Skipped (cached): 1387
Total time: 0.0 min

Cached flow files: 1387
Sample: Knot_Tying_B001_1627_1735.npy
Shape:  (7, 2, 224, 224)  (clip_len-1, 2, H, W)
Min: -562.270, Max: 109.659
Total cache size: 3897.4 MB


### Two stream dataset (RGB + flow)

In [11]:
class JIGSAWSClipDatasetWithFlow(Dataset):
    def __init__(self, task='Suturing', split_trials=None, clip_len=8):
        self.clip_len = clip_len
        self.clips = []  # (video_path, start, end, label, cache_key)

        trans_dir = f"JIGSAWS/{task}/transcriptions/"
        video_dir = f"JIGSAWS/{task}/video/"

        for fname in sorted(os.listdir(trans_dir)):
            if not fname.endswith('.txt'):
                continue
            trial = fname.replace(f'{task}_', '').replace(".txt", '')
            if split_trials and trial not in split_trials:
                continue

            video_path = os.path.join(video_dir, f"{task}_{trial}_capture1.avi")

            if not os.path.exists(video_path):
                continue

            with open(os.path.join(trans_dir, fname)) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 3:
                        continue
                    start, end, gesture = int(parts[0]), int(parts[1]), parts[2]
                    if gesture in GESTURE_MAP and (end - start) >= clip_len:
                        key = f"{task}_{trial}_{start}_{end}.npy"
                        self.clips.append((video_path, start, end, GESTURE_MAP[gesture], key))

        
        self.rgb_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias=True),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        # flow normalization - zero mean, unit std across typical flow magnitudes
        self.flow_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias=True),
        ])

        print(f"{task} {split_trials}: {len(self.clips)} clips")

    def _load_rgb(self, video_path, start, end):
        # load the RGB frames from gesture seg
    
        frame_ids = set(np.linspace(start, end, self.clip_len, dtype=int).tolist())
        frames = []
        container = av.open(video_path)
        for i, frame in enumerate(container.decode(video=0)):
            if i in frame_ids:
                img = frame.to_ndarray(format='rgb24')
                frames.append(self.rgb_transform(img))
            if len(frames) == self.clip_len:
                break
        container.close()
        while len(frames) < self.clip_len:
            frames.append(frames[-1])
        return torch.stack(frames)  # (T, 3, 224, 224)
    
    def _load_flow(self, key):
        # load precomputed flow from cache (T - 1, 2, 224, 224)

        cache_path = os.path.join(FLOW_CACHE_DIR, key)
        flow = np.load(cache_path)  # (7, 2, 224, 224)
        flow = torch.tensor(flow, dtype=torch.float32)

        # normalize flow to roughly [-1, 1] range
        flow = flow/(flow.abs().max() + 1e-6)
        return flow # (T-1, 2, 224, 224)
    
    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        video_path, start, end, label, key = self.clips[idx]
        rgb  = self._load_rgb(video_path, start, end)   # (8, 3, 224, 224)
        flow = self._load_flow(key)       # (7, 2, 224, 224)

        # pad flow to match rgb length by repeating last flow map

        last = flow[-1:].clone()
        flow = torch.cat([flow, last], dim=0)     # (8, 2, 224, 224)
        return rgb, flow, label
    


# sanity check
ds_check = JIGSAWSClipDatasetWithFlow('Suturing')
rgb, flow, label = ds_check[0]
print(f"RGB shape:  {rgb.shape}")
print(f"Flow shape: {flow.shape}")
print(f"Label: {label}, Gesture: {list(GESTURE_MAP.keys())[label]}")

Suturing None: 793 clips
RGB shape:  torch.Size([8, 3, 224, 224])
Flow shape: torch.Size([8, 2, 224, 224])
Label: 0, Gesture: G1


### SAIS Model

In [12]:
import timm
import torch.nn as nn
import torch.nn.functional as F

class SAISModel(nn.Module):
    def __init__(self, n_classes = 10, clip_len=8, feat_dim=384,
                 proj_dim=256, n_heads=6, n_layers=4, dropout=0.1):
        super().__init__()
        self.clip_len = clip_len

        # RGB stream - frozen DINO-ViT-Small
        self.rgb_vit = timm.create_model(
            'vit_small_patch16_224.dino',
            pretrained=True, num_classes=0
        )
        for p in self.rgb_vit.parameters():
            p.requires_grad = False

        # FLOW stream - frozen DINO-ViT-Small
        # flow has 2 channels not 3 so we poject 2ch -> 3ch with a small learned conv before ViT
        self.flow_proj = nn.Conv2d(2, 3, kernel_size=1) # learned projection
        self.flow_vit = timm.create_model(
            'vit_small_patch16_224.dino',
            pretrained=True, num_classes=0
        )
        for p in self.flow_vit.parameters():
            p.requires_grad = False


        # RGB transformer
        self.rgb_cls   = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.rgb_pos   = nn.Parameter(torch.randn(1, clip_len + 1, feat_dim))
        rgb_layer      = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.rgb_transformer = nn.TransformerEncoder(rgb_layer, n_layers)
        self.rgb_norm        = nn.LayerNorm(feat_dim)

        # Flow transformer - separate weights, same architecture
        self.flow_cls  = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.flow_pos  = nn.Parameter(torch.randn(1, clip_len + 1, feat_dim))
        flow_layer     = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.flow_transformer = nn.TransformerEncoder(flow_layer, n_layers)
        self.flow_norm        = nn.LayerNorm(feat_dim)

        # proj head
        self.proj_head = nn.Sequential(
            nn.Linear(feat_dim * 2, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        # prototype classifier
        self.prototypes = nn.Parameter(torch.randn(n_classes, proj_dim))
        self.temperature = 0.07

    def encode_stream(self, x, vit):
        # encode T frames through frozen Vit independently
        # x:   (B, T, C, H, W)
        # out: (B, T, feat_dim)
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        with torch.no_grad():
            x = vit(x)          # (B*T, feat_dim
        return x.view(B, T, -1)     # (B, T, feat_dim)
    

    def temporal_encode(self, x, cls_token, pos_embed, transformer, norm):
        # run temporal transformeer on frame features
        # x:   (B, T, feat_dim)
        # out: (B, feat_dim) — CLS token

        B = x.shape[0]
        cls = cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)   # (B, T+1, feat_dim)
        x   = x + pos_embed
        x   = transformer(x)
        x   = norm(x)
        return x[:, 0, :]     # cls token (B, feat_dim)
    
    def forward(self, rgb, flow, labels=None):
        # rgb:    (B, T, 3, H, W)
        # flow:   (B, T, 2, H, W)
        # labels: (B,) during training, None at inference
        

        B, T, _, H, W = flow.shape

        # proj the 2ch to 3ch
        flow_3ch = self.flow_proj(flow.view(B*T, 2, H, W)) # (B*T, 3, H, W)
        flow_3ch = flow_3ch.view(B, T, 3, H, W)

        # extract rgb and flow feat
        rgb_feat = self.encode_stream(rgb, self.rgb_vit) # (B, T, 384)
        flow_feat = self.encode_stream(flow_3ch, self.flow_vit) # (B, T, 384)

        # separate temporal transformers per stream
        h_rgb  = self.temporal_encode(
            rgb_feat,  self.rgb_cls,  self.rgb_pos,
            self.rgb_transformer,  self.rgb_norm
        )  # (B, 384)
        h_flow = self.temporal_encode(
            flow_feat, self.flow_cls, self.flow_pos,
            self.flow_transformer, self.flow_norm
        )  # (B, 384)

        # fuse by ele wise addn
        fused = torch.cat([h_rgb, h_flow], dim=-1)  # (B, 768)

        # proj + proto classify
        h = self.proj_head(fused)       # (B, proj_dim)
        h_norm = F.normalize(h, dim=1)
        p_norm = F.normalize(self.prototypes, dim=1)
        logits = h_norm @ p_norm.T / self.temperature

        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            return loss, logits
        return logits
    
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SEED)
model4 = SAISModel(n_classes=10, clip_len=8).to(device)

trainable4 = [p for p in model4.parameters() if p.requires_grad]
optimizer4 = torch.optim.Adam(trainable4, lr=1e-4, weight_decay=1e-4)
scheduler4 = torch.optim.lr_scheduler.StepLR(optimizer4, step_size=5, gamma=0.5)

total = sum(p.numel() for p in model4.parameters() if p.requires_grad)
print(f"Trainable params: {total:,}  (both ViTs frozen)")
print(f"Device: {device}")

Trainable params: 14,470,153  (both ViTs frozen)
Device: mps


### DataLoader

In [13]:
from torch.utils.data import DataLoader

train_ds4 = JIGSAWSClipDatasetWithFlow('Suturing', split_trials=train_trials)
test_ds4  = JIGSAWSClipDatasetWithFlow('Suturing', split_trials=test_trials)

train_loader4 = DataLoader(train_ds4, batch_size=8, shuffle=True,  num_workers=0)
test_loader4  = DataLoader(test_ds4,  batch_size=8, shuffle=False, num_workers=0)

print(f"Train clips: {len(train_ds4)}, Test clips: {len(test_ds4)}")
print(f"Train batches: {len(train_loader4)}, Test batches: {len(test_loader4)}")

Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 clips
Suturing ['B001', 'B002', 'B003', 'B004']: 83 clips
Train clips: 710, Test clips: 83
Train batches: 89, Test batches: 11


### Training

In [14]:
from sklearn.metrics import f1_score
import time

print("Stage 4 — Two-Stream SAIS (RGB + Optical Flow) — Suturing")
print("="*50)


best_test_f1 = 0

for epoch in range(1,16):
    t0 = time.time()

    # train 
    model4.train()
    tr_preds, tr_labels_list = [], []
    for rgb, flow, labels in train_loader4:
        rgb, flow, labels = rgb.to(device), flow.to(device), labels.to(device)

        loss, logits = model4(rgb, flow, labels=labels)

        optimizer4.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable4, max_norm=1.0)
        optimizer4.step()

        tr_preds.extend(logits.argmax(1).cpu().numpy())
        tr_labels_list.extend(labels.cpu().numpy())
    scheduler4.step()
    tr_f1 = f1_score(tr_labels_list, tr_preds, average='macro', zero_division=0)

    # eval
    model4.eval()
    te_preds, te_labels_list = [], []
    with torch.no_grad():
        for rgb, flow, labels in test_loader4:
            rgb, flow, labels = rgb.to(device), flow.to(device), labels.to(device)
            logits = model4(rgb, flow)
            te_preds.extend(logits.argmax(1).cpu().numpy())
            te_labels_list.extend(labels.cpu().numpy())
    te_f1 = f1_score(te_labels_list, te_preds, average='macro', zero_division=0)

    if te_f1 > best_test_f1:
        best_test_f1 = te_f1

    print(f"Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f} | {time.time()-t0:.1f}s")

print(f"\nStage 4 Best: {best_test_f1:.3f}")

Stage 4 — Two-Stream SAIS (RGB + Optical Flow) — Suturing
Epoch  1 | Train F1: 0.191 | Test F1: 0.640 | 128.4s
Epoch  2 | Train F1: 0.601 | Test F1: 0.545 | 126.3s
Epoch  3 | Train F1: 0.737 | Test F1: 0.746 | 126.8s
Epoch  4 | Train F1: 0.800 | Test F1: 0.782 | 126.6s
Epoch  5 | Train F1: 0.810 | Test F1: 0.834 | 126.4s
Epoch  6 | Train F1: 0.863 | Test F1: 0.869 | 127.1s
Epoch  7 | Train F1: 0.880 | Test F1: 0.856 | 126.8s
Epoch  8 | Train F1: 0.879 | Test F1: 0.865 | 126.0s
Epoch  9 | Train F1: 0.930 | Test F1: 0.917 | 296.5s
Epoch 10 | Train F1: 0.929 | Test F1: 0.907 | 127.9s
Epoch 11 | Train F1: 0.962 | Test F1: 0.917 | 130.1s
Epoch 12 | Train F1: 0.981 | Test F1: 0.917 | 127.8s
Epoch 13 | Train F1: 0.997 | Test F1: 0.882 | 126.4s
Epoch 14 | Train F1: 0.998 | Test F1: 0.917 | 126.2s
Epoch 15 | Train F1: 0.998 | Test F1: 0.917 | 127.4s

Stage 4 Best: 0.917


In [15]:
per_tasks = ['Knot_Tying', 'Needle_Passing']
task4_results = {}

for task in per_tasks:
    print(f"\n{task}")

    task_train_ds = JIGSAWSClipDatasetWithFlow(task, split_trials=train_trials)
    task_test_ds  = JIGSAWSClipDatasetWithFlow(task, split_trials=test_trials)

    if len(task_test_ds) == 0:
        print("  Skipping — no test clips")
        continue

    task_train_loader = DataLoader(task_train_ds, batch_size=8, shuffle=True,  num_workers=0)
    task_test_loader  = DataLoader(task_test_ds,  batch_size=8, shuffle=False, num_workers=0)

    # fresh model per task
    torch.manual_seed(SEED)
    tm = SAISModel(n_classes=10, clip_len=8).to(device)
    trainable_m = [p for p in tm.parameters() if p.requires_grad]
    to = torch.optim.Adam(trainable_m, lr=1e-4, weight_decay=1e-4)
    ts = torch.optim.lr_scheduler.StepLR(to, step_size=5, gamma=0.5)

    best_f1 = 0
    for epoch in range(1, 16):
        # train
        tm.train()
        tr_preds, tr_labels_list = [], []
        for rgb, flow, labels in task_train_loader:
            rgb, flow, labels = rgb.to(device), flow.to(device), labels.to(device)
            loss, logits = tm(rgb, flow, labels=labels)
            to.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_m, max_norm=1.0)
            to.step()
            tr_preds.extend(logits.argmax(1).cpu().numpy())
            tr_labels_list.extend(labels.cpu().numpy())
        ts.step()
        tr_f1 = f1_score(tr_labels_list, tr_preds, average='macro', zero_division=0)

        # eval
        tm.eval()
        te_preds, te_labels_list = [], []
        with torch.no_grad():
            for rgb, flow, labels in task_test_loader:
                rgb, flow, labels = rgb.to(device), flow.to(device), labels.to(device)
                logits = tm(rgb, flow)
                te_preds.extend(logits.argmax(1).cpu().numpy())
                te_labels_list.extend(labels.cpu().numpy())
        te_f1 = f1_score(te_labels_list, te_preds, average='macro', zero_division=0)

        if te_f1 > best_f1:
            best_f1 = te_f1

        print(f"  Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f}")

    task4_results[task] = best_f1

for task, f1 in task4_results.items():
    print(f"  {task:<20}: Best Test F1 = {f1:.3f}")


Knot_Tying
Knot_Tying ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 51 clips
Knot_Tying ['B001', 'B002', 'B003', 'B004']: 4 clips
  Epoch  1 | Train F1: 0.256 | Test F1: 1.000
  Epoch  2 | Train F1: 0.979 | Test F1: 1.000
  Epoch  3 | Train F1: 0.979 | Test F1: 1.000
  Epoch  4 | Train F1: 1.000 | Test F1: 1.000
  Epoch  5 | Train F1: 1.000 | Test F1: 1.000
  Epoch  6 | Train F1: 1.000 | Test F1: 1.000
  Epoch  7 | Train F1: 1.000 | Test F1: 1.000
  Epoch  8 | Train F1: 1.000 | Test F1: 1.000
  Epoch  9 | Train F1: 1.000 | Test F1: 1.000
  Epoch 10 | Train F1: 1.000 | Test F1: 1.000
  Epoch 11 | Train F1: 1.000 | Test F1: 1.000
  Epoch 12 | Train F1: 1.000 | Test F1: 1.000
  Epoch 13 | Train F1: 1.000 | Test F1: 1.000
  Epoch 14 | Train F1: 1.000 | Test

In [16]:
from torch.utils.data import ConcatDataset

tasks = ['Suturing', 'Knot_Tying', 'Needle_Passing']
train_ds4_combined = ConcatDataset([JIGSAWSClipDatasetWithFlow(t, split_trials=train_trials) for t in tasks])
test_ds4_combined  = ConcatDataset([JIGSAWSClipDatasetWithFlow(t, split_trials=test_trials)  for t in tasks])

train_loader4_combined = DataLoader(train_ds4_combined, batch_size=8, shuffle=True,  num_workers=0)
test_loader4_combined  = DataLoader(test_ds4_combined,  batch_size=8, shuffle=False, num_workers=0)

print(f"Total train clips: {len(train_ds4_combined)}")
print(f"Total test clips:  {len(test_ds4_combined)}")

# fresh model
torch.manual_seed(SEED)
model4_combined = SAISModel(n_classes=10, clip_len=8).to(device)
trainable4_combined = [p for p in model4_combined.parameters() if p.requires_grad]
optimizer4_combined = torch.optim.Adam(trainable4_combined, lr=1e-4, weight_decay=1e-4)
scheduler4_combined = torch.optim.lr_scheduler.StepLR(optimizer4_combined, step_size=5, gamma=0.5)

best_combined_f1 = 0

for epoch in range(1, 16):
    t0 = time.time()

    # train
    model4_combined.train()
    tr_preds, tr_labels_list = [], []
    for rgb, flow, labels in train_loader4_combined:
        rgb, flow, labels = rgb.to(device), flow.to(device), labels.to(device)
        loss, logits = model4_combined(rgb, flow, labels=labels)
        optimizer4_combined.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable4_combined, max_norm=1.0)
        optimizer4_combined.step()
        tr_preds.extend(logits.argmax(1).cpu().numpy())
        tr_labels_list.extend(labels.cpu().numpy())
    scheduler4_combined.step()
    tr_f1 = f1_score(tr_labels_list, tr_preds, average='macro', zero_division=0)

    # evaluate
    model4_combined.eval()
    te_preds, te_labels_list = [], []
    with torch.no_grad():
        for rgb, flow, labels in test_loader4_combined:
            rgb, flow, labels = rgb.to(device), flow.to(device), labels.to(device)
            logits = model4_combined(rgb, flow)
            te_preds.extend(logits.argmax(1).cpu().numpy())
            te_labels_list.extend(labels.cpu().numpy())
    te_f1 = f1_score(te_labels_list, te_preds, average='macro', zero_division=0)

    if te_f1 > best_combined_f1:
        best_combined_f1 = te_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model4_combined.state_dict(),
            'best_f1': best_combined_f1,
        }, 'stage4_combined_best.pt')

    print(f"Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f} | {time.time()-t0:.1f}s")

print(f"\nStage 4 Combined Best Test F1: {best_combined_f1:.3f}")

Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 clips
Knot_Tying ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 51 clips
Needle_Passing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 453 clips
Suturing ['B001', 'B002', 'B003', 'B004']: 83 clips
Knot_Tying ['B001', 'B002', 'B003', 'B